# Bước 06_0b: Baseline Prophet — Đối Chiếu Với LightGBM
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

`06_0_baseline.ipynb` đã so persistence (3 biến thể) — nhưng ở tầm 15 phút, persistence
vốn RẤT MẠNH (bức xạ mặt trời gần như không đổi trong 15 phút), nên so với nó không
chứng minh được đầy đủ giá trị của việc dùng feature thời tiết + LightGBM.

Notebook này thêm 1 baseline THẬT SỰ có ý nghĩa để đối chiếu: **Prophet** — mô hình
time-series chuẩn, được công nhận rộng rãi, KHÔNG dùng feature thời tiết (chỉ học
seasonality ngày/tuần từ chính lịch sử sản lượng). Đây là baseline hợp lệ, không phải
chọn baseline yếu cho có — Prophet là công cụ dự báo chuỗi thời gian phổ biến, dùng thật
trong công nghiệp, mà chính pipeline gốc (`srcs/05_machine_learning/Forcasting_v3/
13_train_prophet_long_term.py`) cũng có 1 bước Prophet riêng.

**Giả thuyết:** Prophet không có input thời tiết thật (`shortwave_radiation`,
`chi_so_troi_quang`...) nên khi trời có mây/biến động sẽ dự báo sai nhiều hơn hẳn
LightGBM — đây mới là điểm chứng minh giá trị thật của việc dùng feature thời tiết.

## 2. Import thư viện và khai báo tham số

In [2]:
import time
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

import numpy as np
import pandas as pd
from prophet import Prophet

# ── Tham số ──
AUDIT_PATH = '../../data/model/v3/06_1_khu_tre_pha/prediction_audit_h1.parquet'
OUTPUT_DIR = '../../data/model/v3/06_0_baseline'
OUTPUT_CSV = f'{OUTPUT_DIR}/prophet_baseline_by_site.csv'

TY_LE_TRAIN = 0.8   # 80% dau chuoi thoi gian moi site de train Prophet, 20% cuoi de test
MIN_DONG_MOI_SITE = 200
MIN_DONG_TEST = 20

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Đã import thư viện và khai báo tham số.")
print(f"- Đọc audit từ : {AUDIT_PATH}")
print(f"- Ghi kết quả ra: {OUTPUT_CSV}")

Đã import thư viện và khai báo tham số.
- Đọc audit từ : ../../data/model/v3/06_1_khu_tre_pha/prediction_audit_h1.parquet
- Ghi kết quả ra: ../../data/model/v3/06_0_baseline/prophet_baseline_by_site.csv


## 3. Đọc dữ liệu — dùng đúng tập test của LightGBM để so sánh công bằng

In [3]:
df = pd.read_parquet(AUDIT_PATH)
print(f"Đang đọc dữ liệu từ: {AUDIT_PATH}")
print(f"Tổng số dòng: {len(df):,}")

# Chi lay dong measured, giong dung phi vi headline chinh thuc cua du an
df = df[df['energy_source'] == 'measured'].copy()
df['ds'] = pd.to_datetime(df['timestamp'])
df['y'] = df['y_true']

sites = sorted(df['site_id'].unique())
print(f"Số site: {len(sites)}")
print(f"Số dòng measured: {len(df):,}")
display(df[['site_id', 'ds', 'y', 'energy_source']].head(3))

Đang đọc dữ liệu từ: ../../data/model/v3/06_1_khu_tre_pha/prediction_audit_h1.parquet
Tổng số dòng: 486,120
Số site: 40
Số dòng measured: 231,608


,site_id,ds,y,energy_source
0,1,2021-12-18 09:30:00,5.6250,measured
1,1,2021-12-18 09:45:00,6.4375,measured
2,1,2021-12-18 10:00:00,9.1875,measured


## 4. Hàm train Prophet 1 site + tính WAPE

In [4]:
def compute_wape(y_true, y_pred):
    """WAPE = tong sai so tuyet doi / tong san luong that, don vi %."""
    denom = np.abs(y_true).sum()
    return float(np.abs(y_true - y_pred).sum() / denom * 100.0) if denom > 0 else np.nan


def train_prophet_1_site(df_site):
    """Train Prophet tren 80% dau chuoi thoi gian, du bao 20% cuoi.

    Prophet CHI hoc tu chinh lich su san luong (cot y) + seasonality ngay/tuan cua no,
    KHONG duoc dua feature thoi tiet nao vao - dung ban chat 1 baseline time-series
    thuan tuy, doi lap voi LightGBM co day du feature thoi tiet that.
    """
    d = df_site.sort_values('ds')
    n = len(d)
    if n < MIN_DONG_MOI_SITE:
        return None

    cut = int(n * TY_LE_TRAIN)
    train = d.iloc[:cut][['ds', 'y']]
    test = d.iloc[cut:][['ds', 'y']]
    if len(test) < MIN_DONG_TEST:
        return None

    model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)
    model.fit(train)
    forecast = model.predict(test[['ds']])

    y_pred = forecast['yhat'].clip(lower=0).to_numpy()
    y_true = test['y'].to_numpy()

    return {
        'n_train': len(train),
        'n_test': len(test),
        'y_true': y_true,
        'y_pred': y_pred,
        'wape': compute_wape(y_true, y_pred),
    }


print("Đã định nghĩa compute_wape và train_prophet_1_site.")

Đã định nghĩa compute_wape và train_prophet_1_site.


## 5. Chạy Prophet cho toàn bộ site

In [5]:
t0 = time.time()
ket_qua_site = []
tong_loi = 0.0
tong_thuc = 0.0
site_loi_chay = []

for s in sites:
    df_site = df[df['site_id'] == s]
    try:
        r = train_prophet_1_site(df_site)
    except Exception as e:
        site_loi_chay.append({'site_id': s, 'loi': str(e)})
        continue
    if r is None:
        continue
    sai_so_tuyet_doi = float(np.abs(r['y_true'] - r['y_pred']).sum())
    thuc_te_tuyet_doi = float(np.abs(r['y_true']).sum())
    tong_loi += sai_so_tuyet_doi
    tong_thuc += thuc_te_tuyet_doi
    ket_qua_site.append({
        'site_id': s, 'n_train': r['n_train'], 'n_test': r['n_test'], 'wape': r['wape'],
    })

thoi_gian_chay = time.time() - t0
pooled_wape = tong_loi / tong_thuc * 100.0 if tong_thuc > 0 else np.nan

print(f"Đã chạy Prophet cho {len(ket_qua_site)}/{len(sites)} site trong {thoi_gian_chay:.1f} giây.")
if site_loi_chay:
    print(f"Số site lỗi (bỏ qua): {len(site_loi_chay)}")
    display(pd.DataFrame(site_loi_chay))

df_ket_qua = pd.DataFrame(ket_qua_site).sort_values('wape', ascending=False)
print(f"\n=== PROPHET POOLED WAPE (toàn bộ site, {int(TY_LE_TRAIN*100)}% cuối làm test) = {pooled_wape:.2f}% ===")

10:54:04 - cmdstanpy - INFO - Chain [1] start processing
10:54:04 - cmdstanpy - INFO - Chain [1] done processing
10:54:04 - cmdstanpy - INFO - Chain [1] start processing
10:54:05 - cmdstanpy - INFO - Chain [1] done processing
10:54:05 - cmdstanpy - INFO - Chain [1] start processing
10:54:05 - cmdstanpy - INFO - Chain [1] done processing
10:54:05 - cmdstanpy - INFO - Chain [1] start processing
10:54:06 - cmdstanpy - INFO - Chain [1] done processing
10:54:06 - cmdstanpy - INFO - Chain [1] start processing
10:54:06 - cmdstanpy - INFO - Chain [1] done processing
10:54:06 - cmdstanpy - INFO - Chain [1] start processing
10:54:06 - cmdstanpy - INFO - Chain [1] done processing
10:54:07 - cmdstanpy - INFO - Chain [1] start processing
10:54:07 - cmdstanpy - INFO - Chain [1] done processing
10:54:07 - cmdstanpy - INFO - Chain [1] start processing
10:54:07 - cmdstanpy - INFO - Chain [1] done processing
10:54:07 - cmdstanpy - INFO - Chain [1] start processing
10:54:08 - cmdstanpy - INFO - Chain [1]

Đã chạy Prophet cho 40/40 site trong 18.0 giây.

=== PROPHET POOLED WAPE (toàn bộ site, 80% cuối làm test) = 60.98% ===


## 6. Bảng chi tiết theo site

In [6]:
print("--- TOP 10 SITE PROPHET TỆ NHẤT ---")
display(df_ket_qua.head(10))

print("\n--- TOP 10 SITE PROPHET TỐT NHẤT ---")
display(df_ket_qua.tail(10))

print(f"\nWAPE trung vị theo site: {df_ket_qua['wape'].median():.2f}%")
print(f"WAPE trung bình theo site: {df_ket_qua['wape'].mean():.2f}%")

--- TOP 10 SITE PROPHET TỆ NHẤT ---


,site_id,n_train,n_test,wape
38,41,4728,1183,124.394257
22,25,4776,1195,74.055086
28,31,4832,1209,68.645906
16,17,4757,1190,67.952065
20,22,4813,1204,67.843200
30,33,4870,1218,66.392842
27,30,4708,1178,66.234141
21,23,4832,1208,65.650375
24,27,5420,1355,64.779628
18,20,4848,1213,64.719580



--- TOP 10 SITE PROPHET TỐT NHẤT ---


,site_id,n_train,n_test,wape
25,28,4478,1120,52.766041
6,7,996,250,51.821674
39,42,4764,1192,51.126000
7,8,4948,1237,50.968396
10,11,5066,1267,49.783619
3,4,4326,1082,45.865255
12,13,4520,1131,45.708155
5,6,4944,1236,45.170027
9,10,4808,1202,44.148473
8,9,4238,1060,40.839240



WAPE trung vị theo site: 60.49%
WAPE trung bình theo site: 60.28%


## 7. Bảng so sánh với LightGBM và Persistence

In [7]:
# Doc lai ket qua LightGBM (06_1) va persistence (06_0) da co san de dat canh nhau
LIGHTGBM_KETQUA = '../../data/model/v3/06_1_khu_tre_pha/ket_qua.json'
PERSISTENCE_CSV = '../../data/model/v3/06_0_baseline/baseline_metrics.csv'

import json

bang_so_sanh = []

if os.path.exists(LIGHTGBM_KETQUA):
    with open(LIGHTGBM_KETQUA, encoding='utf-8') as f:
        kq_lgb = json.load(f)
    wape_lgb = kq_lgb.get('metrics', {}).get('measured_daylight', {}).get('wape')
    bang_so_sanh.append({'model': 'LightGBM (đầy đủ feature thời tiết)', 'wape_%': wape_lgb})

if os.path.exists(PERSISTENCE_CSV):
    per = pd.read_csv(PERSISTENCE_CSV)
    per_h1 = per[(per['scope'] == 'measured_daylight') & (per['horizon'] == 'h1')
                 & (per['baseline_model'] == 'persistence_current')]
    if len(per_h1):
        bang_so_sanh.append({'model': 'Persistence (chép giá trị gần nhất)', 'wape_%': float(per_h1['wape'].iloc[0])})

bang_so_sanh.append({'model': 'Prophet (không có feature thời tiết)', 'wape_%': pooled_wape})

df_so_sanh = pd.DataFrame(bang_so_sanh).sort_values('wape_%')
print("--- BẢNG SO SÁNH CUỐI CÙNG (WAPE h1, càng thấp càng tốt) ---")
display(df_so_sanh)

--- BẢNG SO SÁNH CUỐI CÙNG (WAPE h1, càng thấp càng tốt) ---


,model,wape_%
0,LightGBM (đầy đủ feature thời tiết),17.184914
1,Persistence (chép giá trị gần nhất),20.239403
2,Prophet (không có feature thời tiết),60.977881


### Nhận xét
Prophet chỉ học seasonality ngày/tuần từ lịch sử sản lượng, không có feature thời tiết
thật (`shortwave_radiation`, `chi_so_troi_quang`, `cloud_x_shortwave`...) nên không biết
trước những ngày mây/thời tiết bất thường — dẫn tới WAPE cao hơn hẳn LightGBM. Đây là
bằng chứng định lượng cho giá trị thực sự của việc đưa feature thời tiết + downscale bức
xạ vào mô hình, thay vì chỉ dựa vào tính chu kỳ thời gian thuần túy.

## 8. Export kết quả ra CSV

In [8]:
df_ket_qua.to_csv(OUTPUT_CSV, index=False)
df_so_sanh.to_csv(f'{OUTPUT_DIR}/baseline_comparison_final.csv', index=False)

print("--- HOÀN TẤT ---")
print(f"Đã ghi: {OUTPUT_CSV}")
print(f"Đã ghi: {OUTPUT_DIR}/baseline_comparison_final.csv")

--- HOÀN TẤT ---
Đã ghi: ../../data/model/v3/06_0_baseline/prophet_baseline_by_site.csv
Đã ghi: ../../data/model/v3/06_0_baseline/baseline_comparison_final.csv
